In [1]:
# probe_pt.py — 10 秒搞清 .pt 格式
import torch, json
from pathlib import Path

PT_DIR = Path(r"D:\pythonprojects\practice-github\PPI Experiment\output_embeddings")  # ← 改成你的实际路径

# 1. 找文件(自动尝试命名模式)
patterns = ["*.pt", "*.pth"]
files = sorted(p for pat in patterns for p in PT_DIR.glob(pat))
print(f"[文件] {len(files)} 个, 命名如: {[f.name for f in files[:3]]}")

# 2. 深查前 3 个
for f in files[:3]:
    try:
        obj = torch.load(f, map_location="cpu", weights_only=True)
    except TypeError:
        obj = torch.load(f, map_location="cpu")
    if isinstance(obj, dict):
        print(f"\n{f.name}: dict, keys={list(obj.keys())[:5]}")
        for k, v in obj.items():
            if isinstance(v, dict):
                print(f"  {k}: 子层 {list(v.keys())} → tensor shape {v[max(v.keys())].shape}")
            elif hasattr(v, "shape"):
                print(f"  {k}: tensor {tuple(v.shape)}")
    else:
        print(f"\n{f.name}: 裸张量 {tuple(obj.shape)} dtype={obj.dtype}")

# 3. 覆盖率: 对比 metadata
META = Path("af_output/metadata/monomer_metadata.json")
meta = json.loads(META.read_text())
accs_have = {f.stem for f in files}
covered = sum(1 for a in meta if a in accs_have)
print(f"\n[覆盖] {covered}/{len(meta)} 蛋白有 .pt ({100*covered/len(meta):.1f}%)")
print(f"[缺失] {len(meta) - covered} 个将回退 ESM 现算")


[文件] 20386 个, 命名如: ['A0A024RBG1.pt', 'A0A075B6H7.pt', 'A0A075B6H8.pt']

A0A024RBG1.pt: dict, keys=['label', 'representations', 'mean_representations']
  representations: 子层 [33] → tensor shape torch.Size([181, 1280])
  mean_representations: 子层 [33] → tensor shape torch.Size([1280])

A0A075B6H7.pt: dict, keys=['label', 'representations', 'mean_representations']
  representations: 子层 [33] → tensor shape torch.Size([116, 1280])
  mean_representations: 子层 [33] → tensor shape torch.Size([1280])

A0A075B6H8.pt: dict, keys=['label', 'representations', 'mean_representations']
  representations: 子层 [33] → tensor shape torch.Size([117, 1280])
  mean_representations: 子层 [33] → tensor shape torch.Size([1280])

[覆盖] 11019/11019 蛋白有 .pt (100.0%)
[缺失] 0 个将回退 ESM 现算
